In [1]:
import tensorflow as tf
import numpy as np
import cv2
import os 
from tensorflow.keras import layers, Model

In [2]:
def conv_block(input_tensor, filters, kernel_size, strides):
    x = layers.Conv2D(filters, kernel_size, strides=strides, 
                      padding='valid', use_bias=False)(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x


def separable_conv(input_tensor, filters, kernel_size, strides=1, 
                   pool_size=3, pool_stride=2, padding='same', f2=None):
    x = layers.SeparableConv2D(filters, kernel_size, strides=strides, 
                               padding=padding, use_bias=False)(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    if f2:
        x = layers.SeparableConv2D(f2, kernel_size, strides=strides, 
                                   padding=padding, use_bias=False)(x)
    else:
        x = layers.SeparableConv2D(filters, kernel_size, strides=strides, 
                                   padding=padding, use_bias=False)(x)
    
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size, strides=pool_stride, 
                            padding=padding)(x)
    return x

    

In [3]:
def entry_flow(input_tensor):
    # Block 1
    x = conv_block(input_tensor, 32, 3, 2)
    x = conv_block(x, 64, 3, 1)
    
    # Block 2
    b2 = separable_conv(x, 128, 3, padding='same')
    residual = layers.Conv2D(filters=128, kernel_size=1, strides=2,
                             padding='same', use_bias=False)(x)
    residual = layers.BatchNormalization()(residual)
    x = layers.Add()([b2, residual])
    x = layers.ReLU()(x)

    # Block 3
    b3 = separable_conv(x, 256, 3, padding='same')
    residual = layers.Conv2D(filters=256, kernel_size=1, strides=2,
                             padding='same', use_bias=False)(x)
    residual = layers.BatchNormalization()(residual)
    x = layers.Add()([b3, residual])
    x = layers.ReLU()(x)
    
    # Block 4
    b4 = separable_conv(x, 728, 3, padding='same')
    residual = layers.Conv2D(filters=728, kernel_size=1, strides=2,
                             padding='same', use_bias=False)(x)
    residual = layers.BatchNormalization()(residual)
    x = layers.Add()([b4, residual])
    x = layers.ReLU()(x)
    
    return x




In [4]:
# Middle Flow
def middle_flow(input_tensor, num_blocks=8):
    x = input_tensor
    for _ in range(num_blocks):
        residual = x
        for _ in range(3):
            x = layers.ReLU()(x)
            x = layers.SeparableConv2D(filters=728, kernel_size=3, 
                                       padding='same', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
        x = layers.Add()([x, residual])
    return x
    
    


In [5]:
# Exit Flow
def exit_flow(input_tensor, num_classes=1000):
    x = layers.ReLU()(input_tensor)
    x = separable_conv(x, 728, 3, padding='same', f2=1024)
    
    residual = layers.Conv2D(filters=1024, kernel_size=1, 
                             strides=2, padding='same', use_bias=False)(input_tensor)
    residual = layers.BatchNormalization()(residual)
    x = layers.Add()([x, residual])

    x = layers.SeparableConv2D(filters=1536, kernel_size=3,
                               padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = layers.SeparableConv2D(filters=2048, kernel_size=3,
                               padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)
    output = layers.Dense(num_classes, activation='softmax')(x)
    
    return output


In [6]:
# Complete Xception Model
def build_xception(input_shape=(299, 299, 3), num_classes=1000):
    inputs = layers.Input(shape=input_shape)
    x = entry_flow(inputs)
    x = middle_flow(x)
    outputs = exit_flow(x, num_classes)
    model = Model(inputs, outputs)
    return model

# Instantiate and compile the model
model = build_xception()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 299, 299, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 149, 149, 32  864         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization (BatchNorm  (None, 149, 149, 32  128        ['conv2d[0][0]']                 
 alization)                     )                                                             